In [363]:
import sqlite3

con = sqlite3.connect(':memory:')
c = con.cursor()

In [364]:
import pandas as pd
def panda(data,header):
    df=pd.DataFrame(data=data,columns=[i[0] for i in header]).to_string(index=False)
    return df
con.execute("PRAGMA foreign_keys=ON")

In [365]:
c.execute("""
CREATE TABLE Plane(
PLN INTEGER PRIMARY KEY AUTOINCREMENT,
NAME VARCHAR(12),
Capacity INT,
Location VARCHAR(10)
)
""")

In [366]:
c.execute("""
CREATE TABLE Pilot(
PN INTEGER,
NAME VARCHAR(25),
Address VARCHAR(40)
)
""")

In [367]:
c.execute("""
CREATE TABLE Flight(
FN VARCHAR(6),
PN INTEGER,
PLN INTEGER,
DC VARCHAR(10),
AC VARCHAR(10),
DT INTEGER,
AT INTEGER)
""")

In [368]:
class Plane:
    def __init__(self,PLN:int,Name:str,Capacity:int,Location:str):
        self.PLN=PLN
        self.Name=Name
        self.Capacity=Capacity
        self.Location=Location

In [369]:
def plane_insert(plane:Plane):
    with con:
        c.execute("""
        INSERT INTO Plane (PLN,Name,Capacity,Location)
        VALUES (:PLN,:Name,:Capacity,:Location)""",
        {"PLN":plane.PLN,"Name":plane.Name,"Capacity":plane.Capacity,"Location":plane.Location})

In [370]:
p1=Plane(100,'AIRBUS',300,'ALGER')
p2=Plane(101, 'B737', 250, 'CASA')
p3=Plane(101, 'B737', 220, 'ORAN')

In [371]:
plane_insert(p1)
plane_insert(p2)

In [372]:
plane_insert(p3)

IntegrityError: UNIQUE constraint failed: Plane.PLN

In [399]:
#15-

In [401]:
c.execute("""
SELECT Pilot.Name AS Pilot_name
FROM Pilot
INNER JOIN (
            SELECT PN 
            FROM Plane,Flight
            ON Plane.PLN=Flight.PLN
            WHERE Plane.Name='AIRBUS'
            ) AS AIRBUS_PN
ON Pilot.PN=AIRBUS_PN.PN
""")
print(panda(c.fetchall(),c.description))

Empty DataFrame
Columns: [Pilot_name]
Index: []


# Exercise 2

## Create Author table

In [405]:
c.execute("""
CREATE TABLE Author(
AuthorID INTEGER PRIMARY KEY AUTOINCREMENT,
NAME VARCHAR(100) NOT NULL,
BirthYear INT)
""")

## Create Book table

In [408]:
c.execute("""
CREATE TABLE Book(
BookID INTEGER PRIMARY KEY AUTOINCREMENT,
Title VARCHAR(150) NOT NULL,
AuthorID INT,
PublishedYear INT,
ISBN VARCHAR(20) UNIQUE,
FOREIGN KEY (AuthorID) REFERENCES Author(AuthorID))
""")

## Create Member table

In [411]:
c.execute("""
CREATE TABLE Member(
MemberID INTEGER PRIMARY KEY AUTOINCREMENT,
FullName VARCHAR(100) NOT NULL,
Email VARCHAR(100) UNIQUE NOT NULL,
MembershipDate DATE NOT NULL
)""")

## Create Borrow table

In [414]:
c.execute("""
CREATE TABLE Borrow(
BorrowID INTEGER PRIMARY KEY AUTOINCREMENT,
MemberID INT,
BookID INT,
BorrowDate DATE NOT NULL,
ReturnDate DATE,
FOREIGN KEY (MemberID) REFERENCES Member(MemberID),
FOREIGN KEY (BookID) REFERENCES Book(BookID))
""")

# Exercise 3

## Create Products table

In [418]:
c.execute("""
CREATE TABLE Products (
product_id INT PRIMARY KEY,
product_name VARCHAR(100),
category VARCHAR(50),
unit_price DECIMAL(10, 2))
""")

## Create Sales table

In [421]:
c.execute("""
CREATE TABLE Sales (
sale_id INT PRIMARY KEY,
product_id INT,
quantity_sold INT,
sale_date DATE,
total_price DECIMAL(10, 2),
FOREIGN KEY (product_id) REFERENCES Products(product_id))
""")

## Insert Into Products

In [424]:
c.execute("""
INSERT INTO Products (product_id, product_name, category, unit_price)
VALUES
(101, 'Laptop', 'Electronics', 500.00),
(102, 'Smartphone', 'Electronics', 300.00),
(103, 'Headphones', 'Electronics', 30.00),
(104, 'Keyboard', 'Electronics', 20.00),
(105, 'Mouse', 'Electronics', 15.00)
""")

In [426]:
c.execute("SELECT * FROM Products")
print("Products:")
print(panda(c.fetchall(),c.description))

Products:
 product_id product_name    category  unit_price
        101       Laptop Electronics         500
        102   Smartphone Electronics         300
        103   Headphones Electronics          30
        104     Keyboard Electronics          20
        105        Mouse Electronics          15


## Insert Into Sales

In [429]:
c.execute("""
INSERT INTO Sales (sale_id, product_id, quantity_sold, sale_date, total_price)
VALUES
(1, 101, 5, '2024-01-01', 2500.00),
(2, 102, 3, '2024-01-02', 900.00),
(3, 103, 2, '2024-01-02', 60.00),
(4, 104, 4, '2024-01-03', 80.00),
(5, 105, 6, '2024-01-03', 90.00)
""")

In [431]:
c.execute("SELECT * FROM Sales")
print("Sales:")
print(panda(c.fetchall(),c.description))

Sales:
 sale_id  product_id  quantity_sold  sale_date  total_price
       1         101              5 2024-01-01         2500
       2         102              3 2024-01-02          900
       3         103              2 2024-01-02           60
       4         104              4 2024-01-03           80
       5         105              6 2024-01-03           90


In [433]:
#1-

In [435]:
c.execute("""
SELECT SUM(total_price)
AS total_price
FROM Sales,Products 
ON Products.product_id=Sales.product_id
WHERE category='Electronics'
""")
print(panda(c.fetchall(),c.description))

 total_price
        3630


In [437]:
#2-

In [439]:
c.execute("""
SELECT product_name, (unit_price*quantity_sold) total_price
FROM Sales,Products 
ON Products.product_id=Sales.product_id""")
print(panda(c.fetchall(),c.description))

product_name  total_price
      Laptop         2500
  Smartphone          900
  Headphones           60
    Keyboard           80
       Mouse           90


In [441]:
#3-

In [443]:
c.execute("""
SELECT product_name 
FROM Products,Sales
ON Products.product_id=Sales.product_id
WHERE quantity_sold=(SELECT MAX(quantity_sold) FROM Sales)
""")
print(panda(c.fetchall(),c.description))

product_name
       Mouse


In [444]:
#4-

In [447]:
c.execute("""
SELECT product_name
FROM Products LEFT JOIN Sales
ON Products.product_id=Sales.product_id
WHERE quantity_sold=0 
OR quantity_sold=NULL
""")
print(panda(c.fetchall(),c.description))

Empty DataFrame
Columns: [product_name]
Index: []


In [449]:
#5-

In [451]:
c.execute("""
SELECT SUM(total_price) AS revenue
FROM Sales,Products
ON Products.product_id=Sales.product_id
GROUP BY category""")
print(panda(c.fetchall(),c.description))

 revenue
    3630


In [453]:
#6-

In [455]:
c.execute("""
SELECT category, AVG(unit_price) AS avg_unit_price
FROM Products
GROUP BY category
ORDER BY avg_unit_price DESC
LIMIT 1
""")
print(panda(c.fetchall(),c.description))

   category  avg_unit_price
Electronics           173.0


In [457]:
#7-

In [459]:
c.execute("""
SELECT product_name FROM Products,Sales
ON Products.product_id=Sales.product_id
WHERE quantity_sold>30""")
print(panda(c.fetchall(),c.description))

Empty DataFrame
Columns: [product_name]
Index: []


In [461]:
#8-

In [463]:
c.execute("""
SELECT COUNT(sale_date) AS sales_count
FROM Sales
GROUP BY strftime('%Y','%m',sale_date)""")
print(panda(c.fetchall(),c.description))

 sales_count
           5


In [465]:
#9-

In [467]:
c.execute("""
SELECT sale_id,Sales.product_id,quantity_sold,sale_date,total_price
FROM Products,Sales
ON Products.product_id=Sales.product_id
WHERE product_name LIKE '%Smart%'""")
print(panda(c.fetchall(),c.description))

 sale_id  product_id  quantity_sold  sale_date  total_price
       2         102              3 2024-01-02          900


In [469]:
#10-

In [471]:
c.execute("""
SELECT AVG(quantity_sold) AS avg_quantity
FROM Sales,Products
ON Products.product_id=Sales.product_id
WHERE unit_price>100 
GROUP BY product_name """)
print(panda(c.fetchall(),c.description))

 avg_quantity
          5.0
          3.0
